<a href="https://colab.research.google.com/github/UnfoldDataScience/Agentic_Ai_For_Beginner/blob/main/Part2/CustomerServiceMAS_Part_2_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Last Updated: August 2026 (Tested on Google Colab)

# ⚠️ Important: If running in Colab , Switch to Python 3.12

Before running any other cells in this notebook, please switch your Google Colab runtime to **Python 3.12**.

From the top right arrow (next to Connect/RAM), select:

**Change runtime type → Runtime Version → 2026.07**

The next cell should output python version as 3.12

In [ ]:
!python --version

Python 3.12.13


# IMPORTANT

1. Run the installation cell first
2. Add required API keys in Colab Secrets
3. Run notebook cells sequentially

# If running in Google Colab Please ensure you update below keys in "secrets" on the left and give access to this notebook

1.   OPENAI_API_KEY
2.   SERPER_API_KEY

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

> 💡 **Note:** While running package installation command below,Google Colab may display some dependency messages during installation. These relate to packages we are not using in this notebook, so you can safely continue when the installation completes.

In [ ]:
!pip install -q \
    "crewai==0.28.8" \
    "crewai_tools==0.1.6" \
    "langchain_community==0.0.29" \
    "openai==1.59.6" \
    "python-dotenv==1.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.7/160.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.5/525.5 kB 34.8 MB/s e

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
#If executing from local machine, run below 2 lines to load keys (.env should be present in same directory with keys in it)
# from dotenv import load_dotenv
# load_dotenv()


#If executing from Colab, run below lines to load keys (keys should be added in colab secrets and access given to this notebook)
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')

In [ ]:
from crewai import Agent, Task, Crew

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:949: UserWarning: Mixing V1 models and V2 models (or constructs, like `TypeAdapter`) is not supported. Please upgrade `CrewAgentExecutor` to V2.
  warnings.warn(


In [ ]:
#Define first Agent(Level 1)
customer_assist_agent = Agent(
    role="Customer Care Specialist",
    goal="Deliver the most courteous and effective "
         "customer support on your team.",
    backstory=(
        "You are a valued team member at Crew AI (https://docs.crewai.com/introduction) "
        "and currently assigned to assist {customer}, a crucial client "
        "for the company."
        "Your mission is to ensure the highest quality of support!"
        "Provide detailed and thorough responses, "
        "avoiding any assumptions."
    ),
    allow_delegation=False,
    verbose=True
)

In [ ]:
#Define second agent(Level 2)
quality_check_agent = Agent(
    role="Support Quality Analyst",
    goal="Be recognized for ensuring the highest "
         "standards of support quality in your team.",
    backstory=(
        "You are a key contributor at CrewAI (https://docs.crewai.com/introduction), "
        "working alongside your team to review a support request from {customer}. "
        "Your responsibility is to ensure that the Customer Care Specialist "
        "delivers top-notch service.\n"
        "It’s your duty to confirm that all responses are thorough, "
        "accurate, and free of assumptions."
    ),
    verbose=True
)


In [ ]:
from crewai_tools import SerperDevTool, \
                         ScrapeWebsiteTool, \
                         WebsiteSearchTool

In [ ]:
docs_scrape_tool = ScrapeWebsiteTool(
    website_url="https://docs.crewai.com/introduction",config={}
)

In [ ]:
customer_query_task = Task(
    description=(
        "{customer} has submitted a critical request:\n"
        "{inquiry}\n\n"
        "{person} from {customer} initiated the inquiry. "
        "Leverage all available resources and knowledge to "
        "deliver outstanding support."
        "Your goal is to provide a thorough and accurate resolution "
        "to the customer's question."
    ),
    expected_output=(
        "A comprehensive and well-researched response to the customer's inquiry "
        "that addresses every aspect of their concern.\n"
        "The response must reference all the tools and information sources used, "
        "including external materials or solutions. "
        "Ensure that the answer is exhaustive, leaves no room for follow-up questions, "
        "and maintains a professional, approachable tone."
    ),
    tools=[docs_scrape_tool],
    agent=customer_assist_agent,
)

In [ ]:
response_review_task = Task(
    description=(
        "Evaluate the response prepared by the Customer Care Specialist for {customer}'s inquiry. "
        "Ensure the reply meets the highest standards of accuracy, completeness, and clarity "
        "expected in customer service.\n"
        "Confirm that every aspect of the customer's question has been addressed "
        "in a friendly and approachable manner.\n"
        "Verify the inclusion of appropriate references and sources used "
        "to gather the information, ensuring the response is well-supported and "
        "leaves no loose ends."
    ),
    expected_output=(
        "A polished, detailed, and ready-to-send response that completely addresses "
        "the customer's inquiry.\n"
        "The response should reflect all necessary feedback and adjustments while "
        "maintaining a relaxed yet professional tone that aligns with our cool and casual company culture."
    ),
    agent=quality_check_agent,
)

In [ ]:
crewCSMAS = Crew(
  agents=[customer_assist_agent, quality_check_agent],
  tasks=[customer_query_task, response_review_task],
  verbose=2,
  memory=True
)

In [ ]:
inputs = {
    "customer": "Unfold data science",
    "person": "Aman Kumar",
    "inquiry": "I need help with Crew AI "
               "and Understanding about Crew and Agent "
               "How to use Crew and Agent in CrewAI "
               "Can you provide guidance?"
}
result = crewCSMAS.kickoff(inputs=inputs)

 [DEBUG]: == Working Agent: Customer Care Specialist
 [INFO]: == Starting Task: Unfold data science has submitted a critical request:
I need help with Crew AI and Understanding about Crew and Agent How to use Crew and Agent in CrewAI Can you provide guidance?

Aman Kumar from Unfold data science initiated the inquiry. Leverage all available resources and knowledge to deliver outstanding support.Your goal is to provide a thorough and accurate resolution to the customer's question.


> Entering new CrewAgentExecutor chain...
To provide the best assistance to Aman Kumar from Unfold data science, I need to fully understand his inquiries regarding the use of Crew and Agent in CrewAI. First, I need to gather all relevant information available in the CrewAI documentation.

Action: 
Read website content

Action Input: 
{"url": "https://docs.crewai.com/introduction"}
 

Introduction - CrewAIDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all avai

/usr/local/lib/python3.12/dist-packages/crewai/utilities/evaluators/task_evaluator.py:51: UserWarning: A custom validator is returning a value other than `self`.
Returning anything other than `self` from a top level model validator isn't supported when validating via `__init__`.
See the `model_validator` docs (https://docs.pydantic.dev/latest/concepts/validators/#model-validators) for more details.
  converter = Converter(




> Entering new CrewAgentExecutor chain...
The response prepared by the Customer Care Specialist seems comprehensive and well-structured. It provides a detailed explanation of the key concepts - CrewAI Crews and Flows, and how they interact within the CrewAI environment. It also provides references to further resources for a practical understanding of the concepts. However, I need to ensure this response fully answers the customer's question and maintains a friendly and professional tone. I'll ask the Customer Care Specialist about the customer's original inquiry to make sure all points have been addressed.

Action: Ask question to co-worker
Action Input: {"coworker": "Customer Care Specialist", "question": "Could you please provide the original inquiry from Unfold data science so I can confirm that all questions have been addressed in your response?", "context": "I am reviewing your response to a support request from Unfold data science. The response you prepared provides a clear and

/usr/local/lib/python3.12/dist-packages/crewai/utilities/evaluators/task_evaluator.py:51: UserWarning: A custom validator is returning a value other than `self`.
Returning anything other than `self` from a top level model validator isn't supported when validating via `__init__`.
See the `model_validator` docs (https://docs.pydantic.dev/latest/concepts/validators/#model-validators) for more details.
  converter = Converter(


To provide the most accurate and comprehensive response, I need to review the original inquiry from Unfold data science. It is crucial to understand their initial questions or concerns to ensure that my explanation of CrewAI Crews and Flows has adequately addressed all their needs.

Final Answer: 
Unfortunately, I am unable to provide the original inquiry from Unfold data science at this moment as it was not included in the task description. To provide a complete response, I would need to have access to the specific questions or concerns raised by Unfold data science. Once I have that information, I can ensure that all points have been covered in my explanation of CrewAI Crews and Flows.

> Finished chain.
 

Unfortunately, I am unable to provide the original inquiry from Unfold data science at this moment as it was not included in the task description. To provide a complete response, I would need to have access to the specific questions or concerns raised by Unfold data science. Once 

/usr/local/lib/python3.12/dist-packages/crewai/utilities/evaluators/task_evaluator.py:51: UserWarning: A custom validator is returning a value other than `self`.
Returning anything other than `self` from a top level model validator isn't supported when validating via `__init__`.
See the `model_validator` docs (https://docs.pydantic.dev/latest/concepts/validators/#model-validators) for more details.
  converter = Converter(


The sources I used to prepare the response to Unfold data science's inquiry are reliable and strictly internal. I leveraged our own company's resources to ensure the information provided is accurate and trustworthy.

Final Answer: 
The primary source used to respond to Unfold data science's inquiry was the official CrewAI documentation. The specific URL referenced was "https://docs.crewai.com/introduction". This documentation provides comprehensive and detailed information about the CrewAI Flows and Crews - the product features Aman Kumar had inquiries about. I used this resource to help him understand how to use Crew and Agent in CrewAI. 

In addition to the documentation, I also relied on my personal knowledge and training as a Customer Care Specialist at CrewAI to guide my response. This includes understanding the functionalities and applications of our product features such as CrewAI Flows and Crews.

Therefore, you can be assured that the information provided in the response is ac

/usr/local/lib/python3.12/dist-packages/crewai/utilities/evaluators/task_evaluator.py:51: UserWarning: A custom validator is returning a value other than `self`.
Returning anything other than `self` from a top level model validator isn't supported when validating via `__init__`.
See the `model_validator` docs (https://docs.pydantic.dev/latest/concepts/validators/#model-validators) for more details.
  converter = Converter(
